In [5]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
# ── Kurulum ve Kütüphaneler ──────────────────────────────────────
!pip install xgboost lightgbm -q

import os, cv2, numpy as np, random, warnings
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
from collections import Counter

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             mean_squared_error, mean_absolute_error,
                             r2_score, roc_curve, auc, classification_report)

# İSTEDİĞİN KLASİK VE HİBRİT MODELLER İÇİN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, Model, Input
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')
print("Kütüphaneler başarıyla yüklendi ✅")

In [ ]:
# ── Veri Yolu ve Sınıflar ────────────────────────────────────────
veri_yolu = "/content/drive/MyDrive/YapayZeka_Hap_Projesi/archive/Drug Vision/Data Combined"
siniflar  = sorted([s for s in os.listdir(veri_yolu)
                    if os.path.isdir(os.path.join(veri_yolu, s))])

print(f"Toplam {len(siniflar)} İlaç Sınıfı Bulundu: {siniflar}")

In [ ]:
# ── Data Augmentation (Over-fitting Engelleyici) ─────────────────
# Hedef %96. Abartılı augmentation modeli yorar, hafif tutuyoruz.
def augment(img):
    return [
        cv2.flip(img, 1), # Yatay çevirme (Aynalama)
        cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE), # 90 derece dönüş
        cv2.GaussianBlur(img, (3,3), 0) # Hafif bulanıklaştırma (Gürültü toleransı)
    ]

In [ ]:
# ── Veri Yükleme (Ön İşleme) ─────────────────────────────────────
print("Orijinal veriler yükleniyor... Lütfen bekleyin.")
X_cnn_orig, y_orig = [], []
BOYUT = (96, 96) # MobileNetV2 için ideal minimum boyut

for sinif_adi in siniflar:
    yol = os.path.join(veri_yolu, sinif_adi)
    resim_isimleri = os.listdir(yol)

    for resim_adi in resim_isimleri:
        r = cv2.imread(os.path.join(yol, resim_adi))
        if r is None: continue
        r = cv2.resize(r, BOYUT)

        # CNN için (RGB formatı ve /255 Normalize)
        X_cnn_orig.append(cv2.cvtColor(r, cv2.COLOR_BGR2RGB).astype('float32') / 255.0)
        y_orig.append(sinif_adi)

X_cnn_orig  = np.array(X_cnn_orig)
y_orig      = np.array(y_orig)
print(f"Toplam orijinal görüntü: {len(y_orig)} adet.")

In [ ]:
# ── Train / Test Bölme ve Augmentation Uygulama ──────────────────
le = LabelEncoder()
le.fit(siniflar)
y_orig_num = le.transform(y_orig)

# Test verisini SAF tutuyoruz (Gerçek dünya simülasyonu)
idx = np.arange(len(y_orig))
idx_train, idx_test = train_test_split(
    idx, test_size=0.2, random_state=42, stratify=y_orig)

X_cnn_test  = X_cnn_orig[idx_test]
y_test      = y_orig[idx_test]
y_test_num  = y_orig_num[idx_test]

print("Eğitim setine Data Augmentation (Veri Çoğaltma) uygulanıyor...")
X_cnn_train, y_train = [], []

for i in idx_train:
    # Orijinal resmi BGR'ye geri çevirip çoğaltma işlemi yapıyoruz
    bgr = cv2.cvtColor((X_cnn_orig[i]*255).astype('uint8'), cv2.COLOR_RGB2BGR)

    for r in [bgr] + augment(bgr):
        X_cnn_train.append(cv2.cvtColor(r, cv2.COLOR_BGR2RGB).astype('float32')/255.0)
        y_train.append(y_orig[i])

X_cnn_train  = np.array(X_cnn_train)
y_train      = np.array(y_train)
y_train_num  = le.transform(y_train)

tum_sonuclar = {}
print(f"\nİşlem Tamam. Eğitim Seti: {len(y_train)} | Test Seti: {len(y_test)} ✅")

In [ ]:
# ── Metrik ve Grafik Fonksiyonları (Senin Tema) ──────────────────
def goster_metrikler(model_adi, y_gercek, y_tahmin, renk='Blues'):
    acc  = accuracy_score(y_gercek, y_tahmin)
    y_n  = le.transform(np.array(y_gercek))
    yh_n = le.transform(np.array(y_tahmin))
    mse  = mean_squared_error(y_n, yh_n)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_n, yh_n)
    r2   = r2_score(y_n, yh_n)

    print(f"\n{'='*55}")
    print(f"  {model_adi}")
    print(f"{'='*55}")
    print(f"  Accuracy : %{acc*100:.2f}")
    print(f"  R²       : {r2:.4f}")
    print(f"  MSE      : {mse:.4f}")
    print(f"  RMSE     : {rmse:.4f}")
    print(f"  MAE      : {mae:.4f}")
    print(f"\n{classification_report(y_gercek, y_tahmin, target_names=siniflar)}")

    cm = confusion_matrix(y_gercek, y_tahmin, labels=siniflar)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap=renk,
                xticklabels=siniflar, yticklabels=siniflar)
    plt.title(f'Confusion Matrix — {model_adi}')
    plt.ylabel('Gerçek Hap'); plt.xlabel('Yapay Zeka Tahmini')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout(); plt.show()

    return acc

def goster_roc(model_adi, y_gercek_str, y_score_array):
    y_bin = label_binarize(y_gercek_str, classes=siniflar)
    plt.figure(figsize=(10,7))
    renkler = cycle(plt.cm.tab10.colors)
    auc_degerler = []

    for i, (sinif, renk) in enumerate(zip(siniflar, renkler)):
        if y_bin[:, i].sum() == 0: continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score_array[:, i])
        alan = auc(fpr, tpr)
        auc_degerler.append(alan)
        plt.plot(fpr, tpr, color=renk, lw=1.8, label=f'{sinif} (AUC={alan:.2f})')

    plt.plot([0,1],[0,1],'k--', lw=1)
    plt.xlabel('False Positive Rate (Yanlış Pozitif Oranı)')
    plt.ylabel('True Positive Rate (Doğru Pozitif Oranı)')
    plt.title(f'ROC Curve (Alıcı İşletim Karakteristiği) — {model_adi}\nOrtalama AUC: {np.mean(auc_degerler):.3f}')
    plt.legend(loc='lower right', fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
# ── 1. ANA MODEL: MobileNetV2 (Transfer Learning) ────────────────
print("LİTERATÜR MODELİ 1: MobileNetV2 Transfer Learning Başlatılıyor...")
# Görüntü boyutumuz (96,96,3)

base = MobileNetV2(input_shape=(96,96,3), include_top=False, weights='imagenet')
base.trainable = False # İlk başta donduruyoruz

inp = Input(shape=(96,96,3))
x   = base(inp, training=False)
x   = layers.GlobalAveragePooling2D()(x)
# Bu katman daha sonra Hibrit modellerde Feature (Özellik) çekmek için kullanılacak
x   = layers.Dense(256, activation='relu', name='feat_layer')(x)
x   = layers.Dropout(0.4)(x) # Overfitting engellemek için Dropout (%40 unutma)
out = layers.Dense(len(siniflar), activation='softmax')(x)

model_tl = Model(inp, out)
model_tl.compile(optimizer=Adam(learning_rate=1e-3),
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

# Erken Durdurma (Overfitting'in en büyük düşmanı)
cb = [EarlyStopping(patience=4, restore_best_weights=True, verbose=1),
      ReduceLROnPlateau(factor=0.5, patience=2, verbose=1)]

print("\nAşama 1: Sadece Eklenen Üst Katman Eğitiliyor...")
model_tl.fit(X_cnn_train, y_train_num, epochs=15, batch_size=64,
             validation_split=0.15, callbacks=cb, verbose=1)

print("\nAşama 2: Fine-Tuning (İnce Ayar)... Tüm modeli haplara uyarlıyoruz.")
base.trainable = True
# Çok derinleri bozmamak için sadece son katmanları eğitime açıyoruz
for layer in base.layers[:-30]:
    layer.trainable = False

model_tl.compile(optimizer=Adam(learning_rate=1e-5), # Çok düşük öğrenme hızı (Hassas ayar)
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

model_tl.fit(X_cnn_train, y_train_num, epochs=10, batch_size=32,
             validation_split=0.15, callbacks=cb, verbose=1)

# Sonuçlar
y_score_tl = model_tl.predict(X_cnn_test, verbose=0)
y_pred_tl  = le.inverse_transform(np.argmax(y_score_tl, axis=1))

acc = goster_metrikler("MobileNetV2 (Derin Öğrenme)", y_test, y_pred_tl, 'copper')
goster_roc("MobileNetV2", y_test, y_score_tl)
tum_sonuclar["MobileNetV2"] = acc

In [ ]:
# ── HİBRİT MODELLER İÇİN ÖZELLİK (FEATURE) ÇIKARIMI ──────────────
# Literatürdeki o "CNN + SVM" ve "CNN + kNN" mantığı tam olarak budur.
# Resmi pikseller olarak değil, CNN'in anladığı o üst düzey "Özellikler" olarak alacağız.
print("Derin Öğrenme Modelinden (MobileNetV2) matematiksel özellikler (Features) çekiliyor...")

feat_model = Model(inputs=model_tl.input,
                   outputs=model_tl.get_layer('feat_layer').output)

X_feat_train = feat_model.predict(X_cnn_train, batch_size=64, verbose=0)
X_feat_test  = feat_model.predict(X_cnn_test,  batch_size=64, verbose=0)

# Klasik Makine Öğrenmesi algoritmaları ölçeklenmiş veri sever
scaler_feat = StandardScaler()
X_feat_train_s = scaler_feat.fit_transform(X_feat_train)
X_feat_test_s  = scaler_feat.transform(X_feat_test)
print("Özellik Çıkarımı Tamamlandı. Hibrit Modeller Başlıyor ✅")

In [ ]:
# ── 2. LİTERATÜR MODELİ: CNN + kNN (Hibrit Yaklaşım) ─────────────
print("LİTERATÜR MODELİ 2: CNN Features + kNN Eğitiliyor...")

model_cnn_knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
model_cnn_knn.fit(X_feat_train_s, y_train)

y_pred_cnn_knn  = model_cnn_knn.predict(X_feat_test_s)
y_score_cnn_knn = model_cnn_knn.predict_proba(X_feat_test_s)

acc = goster_metrikler("Hibrit: CNN + kNN", y_test, y_pred_cnn_knn, 'Greens')
goster_roc("Hibrit: CNN + kNN", y_test, y_score_cnn_knn)
tum_sonuclar["CNN + kNN"] = acc

In [ ]:
# ── 3. LİTERATÜR MODELİ: CNN + SVM (Klasik Hibrit) ───────────────
print("LİTERATÜR MODELİ 3: CNN Features + SVM Eğitiliyor...")

model_cnn_svm = SVC(kernel='rbf', C=5, gamma='scale', probability=True, random_state=42)
model_cnn_svm.fit(X_feat_train_s, y_train)

y_pred_cnn_svm  = model_cnn_svm.predict(X_feat_test_s)
y_score_cnn_svm = model_cnn_svm.predict_proba(X_feat_test_s)

acc = goster_metrikler("Hibrit: CNN + SVM", y_test, y_pred_cnn_svm, 'Oranges')
goster_roc("Hibrit: CNN + SVM", y_test, y_score_cnn_svm)
tum_sonuclar["CNN + SVM"] = acc

In [ ]:
# ── Final Karşılaştırma Tablosu ───────────────────────────────────
print("\n" + "🏆"*30)
print("   LİTERATÜR REFERANSLI MODELLER KARŞILAŞTIRMASI")
print("🏆"*30)

sirali = sorted(tum_sonuclar.items(), key=lambda x: x[1], reverse=True)

print(f"\n  {'Model':<38} {'Accuracy':>8}  Bar")
print(f"  {'-'*65}")
for i, (isim, acc) in enumerate(sirali):
    bar   = '█' * int(acc * 35)
    medal = ['🥇','🥈','🥉'][i] if i < 3 else '  '
    print(f"  {medal} {isim:<36} %{acc*100:>5.2f}  {bar}")

# Görsel Karşılaştırma Grafiği
plt.figure(figsize=(12,5))
isimler  = [x[0] for x in sirali]
degerler = [x[1]*100 for x in sirali]
renkler_bar = ['gold','silver','#cd7f32'] + ['steelblue']*(len(sirali)-3)

bars = plt.barh(isimler, degerler, color=renkler_bar[::-1], edgecolor='white')
plt.xlabel('Accuracy Oranı (%)', fontsize=12)
plt.title('🏆 Akademik Referanslı Modeller — Başarı Karşılaştırması', fontsize=14)
plt.xlim(0, 110)

for bar, val in zip(bars, degerler[::-1]):
    plt.text(val+0.5, bar.get_y()+bar.get_height()/2,
             f'%{val:.2f}', va='center', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()

en_iyi = sirali[0]
print(f"\n🥇 EN İYİ BAŞARIM: {en_iyi[0]} → %{en_iyi[1]*100:.2f}")

In [ ]:
# ── Kendi Fotoğrafını Test Et (Canlı Performans) ──────────────────
from google.colab import files
print("\n\n📸 Sınav Vakti! Bilgisayarınızdan bir hap fotoğrafı yükleyin...")
uploaded = files.upload()

for fn in uploaded.keys():
    # 1. Fotoğrafı Oku ve Göster
    resim_bgr = cv2.imread(fn)
    resim_rgb = cv2.cvtColor(resim_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(4,4))
    plt.imshow(resim_rgb)
    plt.title("Yüklenen Gerçek Fotoğraf"); plt.axis('off'); plt.show()

    # 2. Resmi Yapay Zeka Formatına Getir (MobileNetV2 giriş boyutu: 96x96)
    r96 = cv2.resize(resim_bgr, (96,96))
    r96_rgb = cv2.cvtColor(r96, cv2.COLOR_BGR2RGB).astype('float32')/255.0
    r96_batch = np.expand_dims(r96_rgb, axis=0) # Batch formatına (1,96,96,3) çevir

    # 3. MobileNetV2 üzerinden matematiksel özellikleri (Feature) çek
    ozellik_feat = scaler_feat.transform(feat_model.predict(r96_batch, verbose=0))

    # 4. Modellerin Tahminleri
    tl_pred      = le.inverse_transform([np.argmax(model_tl.predict(r96_batch, verbose=0))])[0]
    cnn_knn_pred = model_cnn_knn.predict(ozellik_feat)[0]
    cnn_svm_pred = model_cnn_svm.predict(ozellik_feat)[0]

    print("\n🤖 LİTERATÜR MODELLERİNİN KARARLARI:")
    print(f"  {'─'*45}")
    print(f"  MobileNetV2 (Saf Derin Öğrenme) : {tl_pred.upper()}")
    print(f"  Hibrit Yaklaşım (CNN + kNN)     : {cnn_knn_pred.upper()}")
    print(f"  Klasik Hibrit (CNN + SVM)       : {cnn_svm_pred.upper()}")
    print(f"  {'─'*45}")

    # 5. Konsey (Çoğunluk) Kararı
    tum_tahminler = [tl_pred, cnn_knn_pred, cnn_svm_pred]
    kazanan = Counter(tum_tahminler).most_common(1)[0][0]
    print(f"\n🏆 YAPAY ZEKA KONSEYİ ORTAK KARARI: {kazanan.upper()}")

In [ ]:
# ── MODÜLLERİ DOSYALAMA VE KAYDETME (SAVING MODELS) ─────────────────────
import joblib
import os

# Kayıt klasörü oluşturalım (Drive'da düzenli durması için)
kayit_klasoru = '/content/drive/MyDrive/Hap_Modelleri_V3'
if not os.path.exists(kayit_klasoru):
    os.makedirs(kayit_klasoru)

print(f"📦 Modeller '{kayit_klasoru}' klasörüne kaydediliyor...")

# 1. Derin Öğrenme Modelini Kaydetme (.h5 formatında)
# Sadece ağırlıkları değil, tüm mimariyi kaydeder.
dl_model_yolu = f"{kayit_klasoru}/mobilenet_v3_model.h5"
model_tl.save(dl_model_yolu)
print(" ✅ MobileNetV2 Modeli başarıyla kaydedildi (.h5)")

# 2. Makine Öğrenmesi Modellerini Kaydetme (.joblib formatında)
svm_yolu = f"{kayit_klasoru}/cnn_svm_model.joblib"
knn_yolu = f"{kayit_klasoru}/cnn_knn_model.joblib"
joblib.dump(model_cnn_svm, svm_yolu)
joblib.dump(model_cnn_knn, knn_yolu)
print(" ✅ SVM ve kNN Modelleri başarıyla kaydedildi (.joblib)")

# 3. Yardımcı Aletleri Kaydetme (Tahmin yaparken bunlar şart!)
# Verileri ölçeklendiren (Scaler) ve Etiketleri sayıya çeviren (LabelEncoder)
scaler_yolu = f"{kayit_klasoru}/scaler_feat.joblib"
le_yolu = f"{kayit_klasoru}/label_encoder.joblib"
joblib.dump(scaler_feat, scaler_yolu)
joblib.dump(le, le_yolu)
print(" ✅ Scaler ve LabelEncoder araçları başarıyla kaydedildi (.joblib)")

print("\n🎉 BÜTÜN SİSTEM GÜVENLE YEDEKLENDİ! Artık Colab çökse bile korkmaya gerek yok.")